In [9]:
import json
import openai
import requests

client = openai.OpenAI()

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

In [10]:
# 함수 정의

def get_popular_movies():
    """인기 영화 목록을 가져온다."""
    return requests.get(f"{BASE_URL}/movies").json()

def get_movie_details(id):
    """영화 상세 정보를 가져온다."""
    return requests.get(f"{BASE_URL}/movies/{id}").json()

def get_movie_credits(id):
    """영화 출연진/제작진 정보를 가져온다."""
    return requests.get(f"{BASE_URL}/movies/{id}/credits").json()

# 함수 매핑
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

In [11]:
# AI에게 알려줄 도구(tools) 정의
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of currently popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the cast and crew of a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [12]:
# 대화 이력 (메모리)
messages = []
messages.append(
    {
        "role": "system",
        "content": "You are a movie expert agent. Answer questions about movies using the provided tools. Reply in the same language the user uses.",
    }
)

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    message = response.choices[0].message
    process_ai_response(message)

def process_ai_response(message):
    if message.tool_calls:
        # AI의 tool_calls 응답을 messages에 기록
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments,
                        },
                    }
                    for tc in message.tool_calls
                ],
            }
        )
        # 각 tool_call에 대해 실제 함수 실행
        for tc in message.tool_calls:
            fn_name = tc.function.name
            arguments = tc.function.arguments
            try:
                args = json.loads(arguments)
            except json.JSONDecodeError:
                args = {}
            print(f"Function Calling {fn_name}({args})")

            function_to_run = FUNCTION_MAP.get(fn_name)
            result = function_to_run(**args)

            # 실행 결과를 role: "tool"로 추가
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "name": fn_name,
                    "content": json.dumps(result, ensure_ascii=False),
                }
            )
        # 도구 결과를 포함해서 AI 다시 호출
        call_ai()
    else:
        # 일반 텍스트 응답
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")

In [13]:
print("Movie Agent (종료: q)")
while True:
    user_input = input("\nYou: ")
    if user_input in ("q", "quit"):
        break
    else:
        print(f"User: {user_input}")
    messages.append({"role": "user", "content": user_input})
    call_ai()

Movie Agent (종료: q)
User: 안녕
AI: 안녕하세요! 영화에 대해 궁금한 점이 있으면 무엇이든 물어보세요.
User: 나 SF 영화 좋아하는데 영화 추천좀
Function Calling get_popular_movies({})
AI: 여기 몇 가지 SF 영화를 추천해드릴게요:

1. **Avatar: Fire and Ash**
   - **개요**: 제이크 솔리와 네이티리가 자신들의 아들을 잃은 후 아쉬 사람들로부터 팬도라를 지키기 위해 싸우는 이야기입니다.
   - **개봉일**: 2025-12-17
   - **평점**: 7.4
   - ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

2. **Hoppers**
   - **개요**: 과학자들이 사람의 의식을 리얼한 로봇 동물로 옮기는 기술을 개발하고, 동물과 소통할 수 있는 기회를 잡은 한 소녀의 이야기입니다.
   - **개봉일**: 2026-03-04
   - **평점**: 7.6
   - ![포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)

3. **Project Hail Mary**
   - **개요**: 기억을 잃은 과학자가 우주선에서 자신의 임무를 수행하며 지구를 구하기 위해 고군분투하는 이야기입니다.
   - **개봉일**: 2026-03-15
   - **평점**: 8.2
   - ![포스터](https://image.tmdb.org/t/p/w780/yihdXomYb5kTeSivtFndMy5iDmf.jpg)

4. **The Super Mario Galaxy Movie**
   - **개요**: 마리오와 루이지가 새로운 위협에 맞서기 위해 별을 여행하는 이야기입니다.
   - **개봉일**: 2026-04-01
   - **평점**: 6.9
   - ![포스터](https://image.tmdb.org/t/p/w780